In [1]:
import torch
from torch.utils import data
from torch.utils.data import DataLoader
from torch import nn
import torch.nn.functional as F

import random
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import scipy
from tqdm import trange
from tqdm import tqdm
from datetime import datetime
import sys
import os
import time
from sklearn.metrics import roc_auc_score, f1_score
from IPython.display import clear_output

from transformers import BertModel, BertConfig, PreTrainedTokenizer, BasicTokenizer, BertForTokenClassification, utils
#from bertviz import model_view, head_view

import sklearn
from sklearn.metrics import accuracy_score

from Bio import SeqIO
from io import StringIO, BytesIO

import gc

import warnings
warnings.filterwarnings("ignore")

/home/jovyan/shares/SR006.nfs3/anaconda3/envs/alphaflipon/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#!pip install bertviz

In [3]:
from dna_tokenizer import DNATokenizer, seq2kmer

In [4]:
from tabulate import tabulate
def to_fwf(df, fname):
    content = tabulate(df.values.tolist(), tablefmt="plain")
    open(fname, "w").write(content)

pd.DataFrame.to_fwf = to_fwf

In [5]:
def split_seq(seq, length = 512, pad = 16):
    res = []
    n = len(seq)
    for st in range(0, n, length - pad):
        if st > 0 and st + pad >= n:
            break
        end = min(st+length, n)
        res.append(seq[st:end])
    return res
        
#def stitch_np_seq(np_seqs, pad = 16):
#    res = np.array([])
#    for seq in np_seqs:
#        res = res[:-pad]
#        res = np.concatenate([res,seq])
#    return res

In [6]:
def stitch_np_seq(np_seqs, pad=16):
    # Calculate total length first to pre-allocate array
    total_length = sum(seq.shape[-1] for seq in np_seqs) - pad * (len(np_seqs) - 1)
    
    # Pre-allocate result array
    res = np.empty(total_length, dtype=np_seqs[0].dtype)
    
    # Track current position
    pos = 0
    
    for i, seq in enumerate(np_seqs):
        seq_len = seq.shape[-1]
        if i > 0:
            # Overlap by removing 'pad' elements from previous segment
            pos -= pad
        # Copy current sequence into the result
        #print(pos,pos+seq_len)
        res[pos:pos+seq_len] = seq[0,:]
        pos += seq_len
    
    return res

In [7]:
class PredDataset(data.Dataset):
    def __init__(self, sequence, tokenizer):
        self.pieces = split_seq(seq2kmer(sequence.upper(),6).split(' '), length = 512, pad = 16)
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.pieces)
    
    def __getitem__(self, index):
        sequence = self.pieces[index]
        encoded_k_mers = self.tokenizer.encode_plus(sequence, add_special_tokens=False, max_length=512)["input_ids"]
        return torch.LongTensor(encoded_k_mers)

In [8]:
tokenizer = DNATokenizer.from_pretrained('6-new-12w-0/')

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'DNATokenizer'.


In [ ]:
import os
import pickle
import gc
from Bio import SeqIO
import torch
from torch.utils.data import DataLoader
from transformers import BertConfig, BertForTokenClassification

# ----------------------------
# IO setup & globals
# ----------------------------
pickle_dir = 'predictions_g4'
os.makedirs(pickle_dir, exist_ok=True)

NW_PREFIX = 'NW_'
NW_AGG_FILE = os.path.join(pickle_dir, f'{NW_PREFIX}_all.pickle')

# Load already-computed NW predictions once (so we can skip them fast)
try:
    with open(NW_AGG_FILE, 'rb') as f:
        _existing_NW = pickle.load(f)
    if not isinstance(_existing_NW, dict):
        _existing_NW = {}
except FileNotFoundError:
    _existing_NW = {}

_NW_done = set(_existing_NW.keys())     # names already saved in past runs
_NW_buffer = {}                               # names computed in *this* run (buffered)

def get_pickle_filename(name):
    """Determine the appropriate pickle filename based on the name pattern"""
    if name.startswith('NW_'):
        return os.path.join(pickle_dir, 'NW_all.pickle')
    else:
        return os.path.join(pickle_dir, f'{name}.pickle')

def name_already_processed(name: str) -> bool:
    """Check if a name has already been processed based on existing files/buffer."""
    if name.startswith(NW_PREFIX):
        # For NW pieces, check the aggregated 'done' set (from disk) or current buffer
        return (name in _NW_done) or (name in _NW_buffer)
    else:
        # For non-NW, presence of a per-name pickle indicates completion
        return os.path.exists(get_pickle_filename(name))

def save_prediction(name: str, prediction):
    """Save prediction (buffer NW; write immediately for others)."""
    if name.startswith(NW_PREFIX):
        # Buffer in memory; write once at the end
        _NW_buffer[name] = prediction
    else:
        # For individual files, just save the prediction immediately
        pickle_file = get_pickle_filename(name)
        with open(pickle_file, 'wb') as f:
            pickle.dump(prediction, f)

def flush_NW_buffer():
    """Merge buffered NW predictions into on-disk aggregate (single write)."""
    if not _NW_buffer:
        return  # nothing to do
    merged = dict(_existing_NW)  # start with what was already there
    merged.update(_NW_buffer)    # add everything from this run
    # Single write
    with open(NW_AGG_FILE, 'wb') as f:
        pickle.dump(merged, f)
    # Update in-memory "done" set for completeness (not strictly needed after loop)
    _NW_done.update(_NW_buffer.keys())

# ----------------------------
# Your original pipeline
# ----------------------------
pred_ds = None
pred_dataloader = None
device = 0
torch.cuda.empty_cache()

config = BertConfig.from_pretrained('/home/jovyan/shares/SR006.nfs3/dumerenkov/DNA/Squid/6-new-12w-0/config.json')
dir_to_pretrained_model = f'dnabert_mm_fold_0_kouzine_g4'
model = BertForTokenClassification.from_pretrained(dir_to_pretrained_model, config=config).to(device)
model.eval()

fasta_sequences = SeqIO.parse(open('/home/jovyan/shares/SR006.nfs3/dumerenkov/DNA/Squid/GCF_001194135.2_ASM119413v2_genomic.fna'),'fasta')

try:
    for fasta in fasta_sequences:
        name, input_sequence = fasta.id, str(fasta.seq)
        print(f"Processing: {name}, length: {len(input_sequence)}")

        # Skip if already processed (fast check, no file open for NW)
        if name_already_processed(name):
            print(f"Skipping {name} - already processed")
            continue

        print('Creating dataset')
        del pred_ds
        del pred_dataloader
        gc.collect()

        # Create dataset and dataloader
        pred_ds = PredDataset(input_sequence, tokenizer)
        pred_dataloader = DataLoader(pred_ds, batch_size=1, drop_last=False)

        # Make predictions
        cur_pred = []
        with torch.no_grad():
            for batch in tqdm(pred_dataloader, desc=f"Predicting {name}"):
                batch = batch.to(device)
                outputs = model(batch)['logits']
                outputs = torch.softmax(outputs, dim=-1)[:, :, 1].cpu().numpy()
                cur_pred.append(outputs)

        # Stitch and save prediction (buffer for NW, immediate for others)
        final_prediction = stitch_np_seq(cur_pred)
        save_prediction(name, final_prediction)
        print(f"Buffered{' (NW)' if name.startswith(NW_PREFIX) else ''} / saved prediction for {name}")

finally:
    # Single write for *all* NW pieces processed in this run
    if _NW_buffer:
        print(f"Committing {len(_NW_buffer)} NW piece(s) to {NW_AGG_FILE} in one write...")
        flush_NW_buffer()

print("All sequences processed!")


Processing: NC_068981.1, length: 199874329
Creating dataset


Predicting NC_068981.1: 100%|██████████| 402973/402973 [42:08<00:00, 159.38it/s] 


Buffered / saved prediction for NC_068981.1
Processing: NC_068982.1, length: 192492730
Creating dataset


Predicting NC_068983.1: 100%|██████████| 338823/338823 [35:52<00:00, 157.38it/s] 


Buffered / saved prediction for NC_068983.1
Processing: NC_068984.1, length: 147805816
Creating dataset


Predicting NC_068984.1:  38%|███▊      | 111926/297996 [12:03<14:33, 212.97it/s]

In [ ]:
1